# Labelled datasets

In [1]:
import os
import sys

sys.path.insert(0, "..")
# public dataset, no token needed -- silence the hub's anonymous-access advisories.
# both are read when huggingface_hub is imported, so they must be set first.
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import pandas as pd
import polars as pl
from huggingface_hub import hf_hub_download

from config import INT2NAME, NAME2INT, SHAH_SEEDS
from data.twd.loader import load

## TWD — benchmark splits (local)

In [2]:
sets = []
for seed in SHAH_SEEDS:
    train, test = load("benchmark", seed=seed)
    sets.append(set(pl.concat([train, test])["sentence"].to_list()))

# three seeds are three partitions of one set, not three samples
print("identical sentence set across seeds:", sets[0] == sets[1] == sets[2])

twd = pl.concat([train, test])
print("rows:", len(twd), "| unique sentences:", twd["sentence"].n_unique())
print("years:", twd["year"].min(), "->", twd["year"].max())
print(
    "labels:",
    {
        INT2NAME[r["label"]]: r["count"]
        for r in twd["label"].value_counts().sort("label").to_dicts()
    },
)

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
identical sentence set across seeds: True
rows: 2480 | unique sentences: 2419
years: 1996 -> 2022
labels: {'dovish': 650, 'hawkish': 606, 'neutral': 1224}


## WCB — 24-bank labelled sentences (HuggingFace)

In [3]:
repo = "gtfintechlab/all_annotated_sentences_25000"
wcb = pd.concat(
    [
        pd.read_parquet(
            hf_hub_download(
                repo, f"5768/{sp}-00000-of-00001.parquet", repo_type="dataset"
            )
        )
        for sp in ("train", "val", "test")
    ],
    ignore_index=True,
)
print(
    "raw rows:",
    len(wcb),
    "\nstance labels:",
    wcb["stance_label"].value_counts().to_dict(),
)

# drop irrelevant to map onto 3 classes, drop fomc so the set is purely non-US
wcb = wcb[wcb["stance_label"].isin(NAME2INT)].copy()
print("after dropping irrelevant:", len(wcb))
wcb = wcb[wcb["bank_name"] != "fomc"].copy()
print("after dropping FOMC rows:", len(wcb))

wcb["sentence"] = wcb["sentences"]
wcb["label"] = wcb["stance_label"]
wcb["label_int"] = wcb["label"].map(NAME2INT)

raw rows: 25000 
stance labels: {'neutral': 8737, 'dovish': 8312, 'hawkish': 7097, 'irrelevant': 854}
after dropping irrelevant: 24146
after dropping FOMC rows: 23187


In [4]:
wcb["key"] = wcb["sentence"].str.strip().str.casefold()
dupes = wcb[wcb["key"].duplicated(keep=False)].sort_values("key")
wcb = wcb.drop_duplicates("key").drop(columns="key")
print("final rows:", len(wcb), "| banks:", wcb["bank_name"].nunique())
wcb[["bank_name", "sentence", "label", "label_int"]].to_csv(
    "wcb/wcb_train.csv", index=False
)
print("saved wcb/wcb_train.csv")

final rows: 23182 | banks: 24
saved wcb/wcb_train.csv


## Summary

In [5]:
twd_pct = (
    (twd["label"].value_counts(normalize=True).sort("label")["proportion"] * 100)
    .round(0)
    .to_list()
)
print("twd      :", len(twd), "|", {INT2NAME[i]: p for i, p in enumerate(twd_pct)})
print(
    "wcb_train:",
    len(wcb),
    "|",
    (wcb["label"].value_counts(normalize=True) * 100).round(0).to_dict(),
)

twd      : 2480 | {'dovish': 26.0, 'hawkish': 24.0, 'neutral': 49.0}
wcb_train: 23182 | {'neutral': 36.0, 'dovish': 34.0, 'hawkish': 29.0}
